# QA/QC calibration chains and investigate station spectra

Part 2 of the revised workflow. It loads temporal, spatial, and absolute
coefficients, checks chain closure, extracts station pixels, computes
mean/std and NDVI740, and compares raw and corrected processing stages
against field spectra.

The May 4 closure compares:

- Path A: Westham May 18 → Westham May 4.
- Path B: Westham May 18 → Brunswick May 18 → Brunswick May 4 → Westham May 4.

The missing transform mapping Path A output to Path B output should have
gain near 1 and offset near 0.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import json, os, re, warnings
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from affine import Affine
from IPython.display import display
from rasterio.enums import Resampling
from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window, from_bounds
from shapely.geometry import box, mapping
from sklearn.linear_model import HuberRegressor, LinearRegression
from sklearn.metrics import r2_score

plt.style.use("default")
%config InlineBackend.figure_format = "retina"
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 250)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
STORAGE_ACCOUNT_NAME = "geoanalytics"
CONTAINER_NAME = "wesa-fup-restricted-027817p"
USE_AZURE_CONTAINER = True

if USE_AZURE_CONTAINER:
    try:
        import hatfield_funcs as hf
        hf.config.configure_azure()
        hf.config.configure_data_root_path(CONTAINER_NAME)
        os.environ.setdefault("AZURE_STORAGE_ACCOUNT", STORAGE_ACCOUNT_NAME)
        os.environ.setdefault("GDAL_AZURE_USE_MSI", "YES")
        os.environ.setdefault("GDAL_CACHEMAX", "512")
    except ImportError:
        warnings.warn("hatfield_funcs unavailable; Azure access must already be configured.")

def container_path(path, container_name=CONTAINER_NAME):
    normalized = str(path).replace("\\", "/").lstrip("/")
    return (
        f"/vsiaz/{container_name}/{normalized}"
        if USE_AZURE_CONTAINER
        else str(PROJECT_ROOT / normalized)
    )

print("Project root:", PROJECT_ROOT)

In [ ]:
DATES = ["2026-05-04", "2026-05-18"]
DLS_MODES = ["DLS", "NoDLS"]
SOURCE_BAND_NUMBERS = list(range(1, 12))
SOURCE_BAND_NAMES = [
    "B1", "B2", "B3", "B4", "PAN",
    "B5", "B6", "B7", "B8", "B9", "B10",
]
SPECTRAL_WAVELENGTHS_NM = {
    "B1": 444, "B2": 475, "B3": 531, "B4": 560,
    "B5": 650, "B6": 668, "B7": 705, "B8": 717,
    "B9": 740, "B10": 842,
}
SPECTRAL_BAND_NAMES = list(SPECTRAL_WAVELENGTHS_NM)
BAND_METADATA = pd.DataFrame({
    "source_band": SOURCE_BAND_NUMBERS,
    "band_name": SOURCE_BAND_NAMES,
})
BAND_METADATA["wavelength_nm"] = BAND_METADATA["band_name"].map(SPECTRAL_WAVELENGTHS_NM)
BAND_METADATA["is_pan"] = BAND_METADATA["band_name"].eq("PAN")

ROUNDS = [1,2]
ROUND_DATES = {1:"2026-05-04",2:"2026-05-18"}
DATE_TO_ROUND = {v:k for k,v in ROUND_DATES.items()}
DLS_MODES = ["DLS","NoDLS"]
SITES = ["westham","brunswick"]
ALL_TOUCHED = False
APPLY_NONDEFAULT_SCALE_OFFSET = True
ASSUMED_RASTER_CRS = None
USE_3X3M_QUADRATS = False
NDVI_RED_NM, NDVI_NIR_NM = 668, 740
RED_BAND_NAME = {v:k for k,v in SPECTRAL_WAVELENGTHS_NM.items()}[NDVI_RED_NM]
NIR_BAND_NAME = {v:k for k,v in SPECTRAL_WAVELENGTHS_NM.items()}[NDVI_NIR_NM]
N_CLOSURE_DRAWS = 5000
RANDOM_SEED = 582091

STATIONS_TO_HIGHLIGHT = {
    (1,"westham"):[f"W{i}" for i in range(24,31)],
    (1,"brunswick"):[f"B{i}" for i in range(24,31)],
    (2,"westham"):["W24","W25","W26","W27","W28","W29","W30","W16","W17","W18","W19","W23"],
    (2,"brunswick"):["B24","B25","B26","B27","B28","B29","B30","B17","B20","B21","B22","B23"],
}
ABSOLUTE_OPTIONS = ["none","legacy","recomputed"]
LEGACY_ABSOLUTE = pd.DataFrame({
    "transform_name":"legacy_absolute",
    "band_name":[f"B{i}" for i in range(1,11)],
    "wavelength_nm":[444,475,531,560,650,668,705,717,740,842],
    "gain":[1.721,1.621,1.246,1.305,1.137,1.114,1.157,1.153,1.187,1.210],
    "offset":[-0.053,-0.045,-0.029,-0.029,-0.023,-0.020,-0.030,-0.028,-0.040,-0.051],
})
SAVE_OUTPUTS = True
OUTPUT_DIR = PROJECT_ROOT/"data/processed/spectra/calibration_chain_qaqc"

## 2. Input paths

In [ ]:
POLYGON_RELATIVE_PATHS = {
    "westham": Path("data/processed/imagery/radiometry_eval/spectrally_constant_features_westham_v2.geojson"),
    "brunswick": Path("data/processed/imagery/radiometry_eval/spectrally_constant_features_brunswick_v2.geojson"),
}
RASTER_RELATIVE_PATHS = {
    ("2026-05-04", "brunswick", "DLS"): Path("airborne_data_20260504_v2_COG/RTB2_10Band_Brunswick_20260504_DLS_v2.tif"),
    ("2026-05-04", "brunswick", "NoDLS"): Path("airborne_data_20260504_v2_COG/RTB2_10Band_Brunswick_20260504_NoDLS_v2.tif"),
    ("2026-05-04", "westham", "DLS"): Path("airborne_data_20260504_v2_COG/RTB2_10Band_Westham_20260504_DLS_v2.tif"),
    ("2026-05-04", "westham", "NoDLS"): Path("airborne_data_20260504_v2_COG/RTB2_10Band_Westham_20260504_NoDLS_v2.tif"),
    ("2026-05-18", "brunswick", "DLS"): Path("airborne_data_20260518_v2_COG/RTB2_10Band_Brunswick_20260518_DLS_v2.tif"),
    ("2026-05-18", "brunswick", "NoDLS"): Path("airborne_data_20260518_v2_COG/RTB2_10Band_Brunswick_20260518_NoDLS_v2.tif"),
    ("2026-05-18", "westham", "DLS"): Path("airborne_data_20260518_v2_COG/RTB2_10Band_Westham_20260518_DLS_v2.tif"),
    ("2026-05-18", "westham", "NoDLS"): Path("airborne_data_20260518_v2_COG/RTB2_10Band_Westham_20260518_NoDLS_v2.tif"),
}
RASTER_PATHS = {k: container_path(v) for k, v in RASTER_RELATIVE_PATHS.items()}
POLYGON_PATHS = {k: container_path(v) for k, v in POLYGON_RELATIVE_PATHS.items()}
display(pd.DataFrame([
    {"flight_date": d, "site": s, "dls_mode": m, "path": p}
    for (d, s, m), p in RASTER_PATHS.items()
]))

STATION_POLYGON_PATH = container_path(
    "data/raw_clean/gnss/biofilm_2x2m_and_3x3m_quadrat_polygons.geojson"
)
TEMPORAL_OUTPUT_DIR = PROJECT_ROOT/"data/processed/imagery/radiometry_eval/polygon_pixel_spectra"
CALIBRATION_OUTPUT_DIR = PROJECT_ROOT/"data/processed/imagery/radiometry_eval/spatial_absolute_calibration"
GROUND_TRUTH_TABLE = None
GROUND_TRUTH_SEARCH_ROOTS = [
    PROJECT_ROOT/"data/processed/spectra",
    PROJECT_ROOT/"data/raw_clean/spectra",
]

## 3. Transform algebra

In [ ]:
def normalize_coef(df, name=None):
    x = df.rename(columns={"slope":"gain","intercept":"offset","band":"band_name","date":"flight_date","mode":"dls_mode"}).copy()
    if not {"band_name","gain","offset"}.issubset(x.columns):
        raise KeyError("Coefficient table needs band_name, gain, offset.")
    if "wavelength_nm" not in x: x["wavelength_nm"] = x.band_name.map(SPECTRAL_WAVELENGTHS_NM)
    if name is not None: x["transform_name"] = name
    elif "transform_name" not in x: x["transform_name"] = "unnamed"
    return x

def compose(first, second, name):
    a,b = normalize_coef(first),normalize_coef(second)
    m=a[["band_name","wavelength_nm","gain","offset"]].merge(
        b[["band_name","wavelength_nm","gain","offset"]],
        on=["band_name","wavelength_nm"],suffixes=("_a","_b"),validate="one_to_one"
    )
    return pd.DataFrame({
        "transform_name":name,"band_name":m.band_name,"wavelength_nm":m.wavelength_nm,
        "gain":m.gain_a*m.gain_b,"offset":m.offset_a*m.gain_b+m.offset_b,
    })

def invert(t,name):
    x=normalize_coef(t)
    return pd.DataFrame({
        "transform_name":name,"band_name":x.band_name,"wavelength_nm":x.wavelength_nm,
        "gain":1/x.gain,"offset":-x.offset/x.gain,
    })

def identity(name="identity", include_pan=True):
    bands=SOURCE_BAND_NAMES if include_pan else SPECTRAL_BAND_NAMES
    return pd.DataFrame({
        "transform_name":name,"band_name":bands,
        "wavelength_nm":[SPECTRAL_WAVELENGTHS_NM.get(b) for b in bands],
        "gain":1.0,"offset":0.0,
    })

def relative(path_a,path_b,name):
    return compose(invert(path_a,"inverse_A"),path_b,name)

def apply_transform(df,t,bands=None):
    out=df.copy(); t=normalize_coef(t).set_index("band_name")
    for band in (bands or [b for b in t.index if b in out]):
        out[band]=t.loc[band,"gain"]*out[band]+t.loc[band,"offset"]
    return out

## 4. Discover temporal coefficients produced by the previous notebook

In [ ]:
def read_table(path):
    return pd.read_csv(path) if Path(path).suffix.lower()==".csv" else pd.read_parquet(path)

def discover_coef_tables(root):
    out=[]
    if root.exists():
        for p in root.rglob("*"):
            if p.suffix.lower() not in {".csv",".parquet",".pq"}: continue
            try: t=read_table(p)
            except Exception: continue
            c={str(x).lower() for x in t.columns}
            if ("gain" in c or "slope" in c) and ("offset" in c or "intercept" in c) and ("band_name" in c or "band" in c):
                out.append((p,t))
    return out

def infer_labels(text):
    s=str(text).lower().replace("-","")
    site="brunswick" if "brunswick" in s else "westham" if "westham" in s else None
    mode="NoDLS" if "nodls" in s else "DLS" if "dls" in s else None
    direction=(
        "may4_to_may18" if ("may4" in s and "may18" in s and s.find("may4")<s.find("may18"))
        else "may18_to_may4" if ("may4" in s and "may18" in s)
        else "may4_to_may18"
    )
    return site,mode,direction

temporal_groups=[]
for path,table in discover_coef_tables(TEMPORAL_OUTPUT_DIR):
    x=normalize_coef(table)
    group_cols=[c for c in ["site","dls_mode","transform_name","source_date","destination_date","from_date","to_date"] if c in x]
    groups=x.groupby(group_cols,dropna=False) if group_cols else [((),x)]
    for key,g in groups:
        site,mode,direction=infer_labels(str(path)+" "+str(key)+" "+" ".join(g.transform_name.astype(str).unique()))
        if site is None and "site" in g and g.site.nunique()==1: site=str(g.site.iloc[0]).lower()
        if mode is None and "dls_mode" in g and g.dls_mode.nunique()==1: mode=str(g.dls_mode.iloc[0])
        if site in SITES and mode in DLS_MODES:
            y=g.copy(); y["site"],y["dls_mode"],y["direction"],y["source_path"]=site,mode,direction,str(path)
            temporal_groups.append(y)

if not temporal_groups:
    raise FileNotFoundError(f"No temporal coefficients under {TEMPORAL_OUTPUT_DIR}")
temporal_all=pd.concat(temporal_groups,ignore_index=True)

def temporal(site,mode,direction="may4_to_may18"):
    x=temporal_all.loc[(temporal_all.site.eq(site))&(temporal_all.dls_mode.eq(mode))&(temporal_all.direction.eq(direction))].copy()
    if x.empty:
        rev="may18_to_may4" if direction=="may4_to_may18" else "may4_to_may18"
        return invert(temporal(site,mode,rev),f"{site}_{direction}_{mode}")
    x["pref"]=x.transform_name.astype(str).str.lower().str.contains("band|per_band",regex=True).astype(int)
    path=x[["source_path","pref"]].drop_duplicates().sort_values("pref",ascending=False).iloc[0].source_path
    x=x.loc[x.source_path.eq(path)].drop_duplicates("band_name").copy()
    return normalize_coef(x,f"{site}_{direction}_{mode}")

TEMPORAL={(s,m):temporal(s,m) for s in SITES for m in DLS_MODES}
display(temporal_all[["source_path","site","dls_mode","direction","band_name","gain","offset"]].drop_duplicates())

## 5. Load spatial and absolute coefficients

In [ ]:
spatial_all=normalize_coef(pd.read_csv(CALIBRATION_OUTPUT_DIR/"all_spatial_coefficients.csv"))
absolute_all=normalize_coef(pd.read_csv(CALIBRATION_OUTPUT_DIR/"westham_may18_absolute_coefficients.csv"))

def spatial(date,mode):
    x=spatial_all.loc[(spatial_all.flight_date.eq(date))&(spatial_all.dls_mode.eq(mode))].copy()
    if x.empty: raise KeyError((date,mode))
    return normalize_coef(x)

def absolute(option,mode):
    if option=="none": return identity("no_absolute",include_pan=False)
    if option=="legacy": return normalize_coef(LEGACY_ABSOLUTE)
    if option=="recomputed":
        return normalize_coef(absolute_all.loc[absolute_all.dls_mode.eq(mode)].copy())
    raise ValueError(option)

## 6. Chain closure coefficients

In [ ]:
chain_rows=[]
closure={}
for mode in DLS_MODES:
    w4_w18=TEMPORAL[("westham",mode)]
    b4_b18=TEMPORAL[("brunswick",mode)]
    b4_w4=spatial("2026-05-04",mode)
    b18_w18=spatial("2026-05-18",mode)

    path_a=invert(w4_w18,f"path_A_w18_to_w4_{mode}")
    path_b=compose(
        compose(invert(b18_w18,"w18_to_b18"),invert(b4_b18,"b18_to_b4"),"w18_to_b4"),
        b4_w4,f"path_B_w18_to_w4_{mode}"
    )
    c=relative(path_a,path_b,f"closure_A_to_B_{mode}")
    c["dls_mode"]=mode; closure[mode]=c
    m=path_a[["band_name","gain","offset"]].merge(
        path_b[["band_name","gain","offset"]],on="band_name",suffixes=("_A","_B")
    ).merge(c[["band_name","gain","offset"]].rename(columns={"gain":"closure_gain","offset":"closure_offset"}),on="band_name")
    m["dls_mode"]=mode; chain_rows.append(m)
chain_comparison=pd.concat(chain_rows,ignore_index=True)
display(chain_comparison)

## 7. Closure uncertainty

In [ ]:
def se(x,column):
    if f"{column}_se" in x: return x[f"{column}_se"].fillna(0).to_numpy(float)
    if f"{column}_ci_low" in x and f"{column}_ci_high" in x:
        return (x[f"{column}_ci_high"]-x[f"{column}_ci_low"]).fillna(0).to_numpy(float)/(2*1.96)
    return np.zeros(len(x))

def draw(t,rng):
    x=normalize_coef(t)
    out=x[["band_name","wavelength_nm"]].copy()
    out["gain"]=rng.normal(x.gain,se(x,"gain"))
    out["offset"]=rng.normal(x.offset,se(x,"offset"))
    out["transform_name"]="draw"
    return out[["transform_name","band_name","wavelength_nm","gain","offset"]]

rng=np.random.default_rng(RANDOM_SEED)
frames=[]
for mode in DLS_MODES:
    for draw_id in range(N_CLOSURE_DRAWS):
        wa=draw(TEMPORAL[("westham",mode)],rng)
        bt=draw(TEMPORAL[("brunswick",mode)],rng)
        s4=draw(spatial("2026-05-04",mode),rng)
        s18=draw(spatial("2026-05-18",mode),rng)
        pa=invert(wa,"A")
        pb=compose(compose(invert(s18,"w18_b18"),invert(bt,"b18_b4"),"w18_b4"),s4,"B")
        c=relative(pa,pb,"closure"); c["dls_mode"],c["draw_id"]=mode,draw_id
        frames.append(c)
closure_draws=pd.concat(frames,ignore_index=True)
closure_summary=closure_draws.groupby(["dls_mode","band_name"],observed=True).agg(
    wavelength_nm=("wavelength_nm","first"),
    gain_mean=("gain","mean"),gain_se=("gain","std"),
    gain_ci_low=("gain",lambda x:x.quantile(.025)),gain_ci_high=("gain",lambda x:x.quantile(.975)),
    offset_mean=("offset","mean"),offset_se=("offset","std"),
    offset_ci_low=("offset",lambda x:x.quantile(.025)),offset_ci_high=("offset",lambda x:x.quantile(.975)),
).reset_index()
display(closure_summary)

fig,axes=plt.subplots(2,1,figsize=(12,8),sharex=True)
for mode,g in closure_summary.groupby("dls_mode"):
    g=g.sort_values("wavelength_nm")
    axes[0].errorbar(g.wavelength_nm,g.gain_mean,
        yerr=[g.gain_mean-g.gain_ci_low,g.gain_ci_high-g.gain_mean],marker="o",capsize=3,label=mode)
    axes[1].errorbar(g.wavelength_nm,g.offset_mean,
        yerr=[g.offset_mean-g.offset_ci_low,g.offset_ci_high-g.offset_mean],marker="o",capsize=3,label=mode)
axes[0].axhline(1,color="black",ls="--"); axes[1].axhline(0,color="black",ls="--")
axes[0].set_ylabel("Closure gain"); axes[1].set_ylabel("Closure offset")
axes[1].set_xlabel("Wavelength (nm)"); axes[0].legend(); axes[1].legend()
fig.tight_layout(); plt.show()

## 8. Empirical closure in the geographic overlap

This uses the selected May 18 overlap patches as actual Westham May 18
anchor spectra. Each patch spectrum is propagated through Path A and Path B.
The resulting Path-B-versus-Path-A fit is the empirical missing transform in
the overlap and should be close to gain 1 and offset 0.

In [ ]:
empirical_closure_rows = []

for mode in DLS_MODES:
    patch_path = (
        CALIBRATION_OUTPUT_DIR
        / f"20260518_{mode}_homogeneous_overlap_patches.geojson"
    )
    patches = gpd.read_file(patch_path)
    anchor = pd.DataFrame({
        band: patches[f"west_{band}_median"]
        for band in SOURCE_BAND_NAMES
    })

    path_a = invert(
        TEMPORAL[("westham", mode)],
        f"path_A_w18_to_w4_{mode}",
    )
    path_b = compose(
        compose(
            invert(
                spatial("2026-05-18", mode),
                f"w18_to_b18_{mode}",
            ),
            invert(
                TEMPORAL[("brunswick", mode)],
                f"b18_to_b4_{mode}",
            ),
            f"w18_to_b4_{mode}",
        ),
        spatial("2026-05-04", mode),
        f"path_B_w18_to_w4_{mode}",
    )

    values_a = apply_transform(anchor, path_a, SOURCE_BAND_NAMES)
    values_b = apply_transform(anchor, path_b, SOURCE_BAND_NAMES)

    for band in SOURCE_BAND_NAMES:
        x = values_a[band].to_numpy(dtype=float)
        y = values_b[band].to_numpy(dtype=float)
        valid = np.isfinite(x) & np.isfinite(y)
        model = LinearRegression().fit(x[valid, None], y[valid])
        pred = model.predict(x[valid, None])
        empirical_closure_rows.append({
            "dls_mode": mode,
            "band_name": band,
            "wavelength_nm": SPECTRAL_WAVELENGTHS_NM.get(band),
            "gain": float(model.coef_[0]),
            "offset": float(model.intercept_),
            "r2": r2_score(y[valid], pred),
            "rmse_between_paths": float(
                np.sqrt(np.mean((y[valid] - x[valid])**2))
            ),
            "mean_difference_pathB_minus_pathA": float(
                np.mean(y[valid] - x[valid])
            ),
            "patch_count": int(valid.sum()),
        })

empirical_closure = pd.DataFrame(empirical_closure_rows)
display(empirical_closure)

## 8. Extract station pixels

In [ ]:
def extraction_crs(src):
    if ASSUMED_RASTER_CRS is not None: return ASSUMED_RASTER_CRS
    if src.crs is None: raise ValueError("Raster has no CRS")
    return src.crs

def read_scaled(src,indexes,window=None):
    d=src.read(indexes=indexes,window=window,masked=True)
    v=np.ma.getdata(d).astype(float); mask=np.ma.getmaskarray(d)
    scales=np.asarray(src.scales)[np.asarray(indexes)-1].astype(float)
    offsets=np.asarray(src.offsets)[np.asarray(indexes)-1].astype(float)
    if APPLY_NONDEFAULT_SCALE_OFFSET and (not np.allclose(scales,1) or not np.allclose(offsets,0)):
        v=v*scales[:,None,None]+offsets[:,None,None]
    v[mask]=np.nan
    return v

def extract_pixels(path,polygons,metadata):
    frames=[]
    with rasterio.open(path) as src:
        p=polygons.to_crs(extraction_crs(src))
        for _,f in p.iterrows():
            try: w=geometry_window(src,[mapping(f.geometry)],boundless=False)
            except rasterio.errors.WindowError: continue
            v=read_scaled(src,SOURCE_BAND_NUMBERS,w)
            inside=geometry_mask([mapping(f.geometry)],out_shape=(int(w.height),int(w.width)),
                                 transform=src.window_transform(w),invert=True,all_touched=ALL_TOUCHED)
            valid=inside & np.isfinite(v).all(axis=0)
            rr,cc=np.where(valid)
            if not len(rr): continue
            df=pd.DataFrame(v[:,rr,cc].T,columns=SOURCE_BAND_NAMES)
            df["station_name"],df["pixel_id"]=f.station_name,np.arange(len(df))
            for k,val in metadata.items(): df[k]=val
            frames.append(df)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

stations=gpd.read_file(STATION_POLYGON_PATH)
stations["station_name"]=stations.station_name.astype("string").str.strip()
stations["round"]=pd.to_numeric(stations["round"],errors="raise").astype("Int64")
stations["is_3x3m"]=stations.is_3x3m.astype("string").str.lower().map(
    {"true":True,"false":False,"yes":True,"no":False,"1":True,"0":False}
).astype("boolean")
stations=stations.loc[stations["round"].isin(ROUNDS)&stations.is_3x3m.eq(USE_3X3M_QUADRATS)].copy()
stations["site"]=np.where(stations.station_name.str.startswith("W"),"westham",
                  np.where(stations.station_name.str.startswith("B"),"brunswick",None))
stations=stations.loc[stations.site.isin(SITES)].copy()
stations["flight_date"]=stations["round"].map(ROUND_DATES)

frames=[]
for date in ROUND_DATES.values():
    rnd=DATE_TO_ROUND[date]
    for site in SITES:
        p=stations.loc[(stations["round"].eq(rnd))&(stations.site.eq(site))]
        for mode in DLS_MODES:
            x=extract_pixels(RASTER_PATHS[(date,site,mode)],p,{
                "flight_date":date,"round":rnd,"site":site,"dls_mode":mode
            })
            x["highlight"]=x.station_name.isin(STATIONS_TO_HIGHLIGHT.get((rnd,site),[]))
            frames.append(x)
station_pixels_raw=pd.concat(frames,ignore_index=True)
print("Raw station pixels:",f"{len(station_pixels_raw):,}")

## 9. Apply routes and absolute choices

In [ ]:
def route_to_w18(date,site,mode,route="preferred"):
    if date=="2026-05-18" and site=="westham": return identity()
    if date=="2026-05-18" and site=="brunswick": return spatial(date,mode)
    if date=="2026-05-04" and site=="westham": return TEMPORAL[(site,mode)]
    if date=="2026-05-04" and site=="brunswick":
        if route=="preferred":
            return compose(TEMPORAL[("brunswick",mode)],spatial("2026-05-18",mode),f"b4_b18_w18_{mode}")
        if route=="alternate":
            return compose(spatial("2026-05-04",mode),TEMPORAL[("westham",mode)],f"b4_w4_w18_{mode}")
    raise KeyError((date,site,mode,route))

def absolute_transform(option,mode):
    if option=="none": return identity("none",include_pan=False)
    if option=="legacy": return normalize_coef(LEGACY_ABSOLUTE)
    if option=="recomputed": return normalize_coef(absolute_all.loc[absolute_all.dls_mode.eq(mode)])
    raise ValueError(option)

frames=[]
for (date,site,mode),raw in station_pixels_raw.groupby(["flight_date","site","dls_mode"],observed=True):
    r=raw.copy(); r["scope_stage"],r["route"],r["absolute_option"]="raw","raw","none"; frames.append(r)
    routes=["preferred","alternate"] if (date=="2026-05-04" and site=="brunswick") else ["preferred"]
    for route in routes:
        w18=apply_transform(raw,route_to_w18(date,site,mode,route),SOURCE_BAND_NAMES)
        w18["scope_stage"],w18["route"],w18["absolute_option"]="westham_may18_scope",route,"none"
        frames.append(w18.copy())
        for option in ["legacy","recomputed"]:
            f=apply_transform(w18,absolute_transform(option,mode),SPECTRAL_BAND_NAMES)
            f["scope_stage"],f["route"],f["absolute_option"]="absolute",route,option
            frames.append(f)
station_pixels=pd.concat(frames,ignore_index=True)
den=station_pixels[NIR_BAND_NAME]+station_pixels[RED_BAND_NAME]
station_pixels["ndvi740"]=np.where(abs(den)>1e-12,(station_pixels[NIR_BAND_NAME]-station_pixels[RED_BAND_NAME])/den,np.nan)
ids=["flight_date","round","site","dls_mode","station_name","highlight","scope_stage","route","absolute_option"]
spectra=(station_pixels.melt(id_vars=ids+["pixel_id"],value_vars=SPECTRAL_BAND_NAMES,
    var_name="band_name",value_name="reflectance")
    .groupby(ids+["band_name"],observed=True).reflectance
    .agg(pixel_count="count",imagery_mean="mean",imagery_std="std",imagery_median="median").reset_index())
spectra["wavelength_nm"]=spectra.band_name.map(SPECTRAL_WAVELENGTHS_NM)
ndvi=(station_pixels.groupby(ids,observed=True).ndvi740
      .agg(pixel_count="count",imagery_ndvi740_mean="mean",imagery_ndvi740_std="std").reset_index())
display(spectra.head()); display(ndvi.head())

## 10. Load field spectra, interpolate to band centers, and compute NDVI740

In [ ]:
def discover_ground_truth():
    candidates=[]
    for root in GROUND_TRUTH_SEARCH_ROOTS:
        if root.exists():
            for p in root.rglob("*"):
                if p.suffix.lower() in {".csv",".parquet",".pq"} and (
                    "ground" in p.name.lower() or "field" in p.name.lower() or "spectra" in p.name.lower()
                ) and "imagery_comparison" not in str(p).lower():
                    candidates.append(p)
    return candidates

def find_col(cols,candidates):
    lookup={str(c).lower():c for c in cols}
    return next((lookup[x.lower()] for x in candidates if x.lower() in lookup),None)

def normalize_ground(df):
    sc=find_col(df.columns,["station_name","station","site_id","plot","name"])
    rc=find_col(df.columns,["round","survey_round","campaign_round"])
    tc=find_col(df.columns,["sample_type","type","measurement_type"])
    wc=find_col(df.columns,["wavelength_nm","wavelength","wl","lambda_nm"])
    vc=find_col(df.columns,["reflectance","mean_reflectance","value","mean"])
    if None in (sc,rc,wc,vc): raise KeyError("Missing station/round/wavelength/reflectance")
    out=pd.DataFrame({
        "station_name":df[sc].astype("string").str.strip(),
        "round":pd.to_numeric(df[rc],errors="coerce"),
        "wavelength_nm":pd.to_numeric(df[wc],errors="coerce"),
        "field_reflectance":pd.to_numeric(df[vc],errors="coerce"),
        "sample_type":df[tc].astype("string").str.strip() if tc else "R",
    }).dropna(subset=["station_name","round","wavelength_nm","field_reflectance"])
    return out.loc[out.sample_type.eq("R")]

if GROUND_TRUTH_TABLE is None:
    compatible=[]
    for p in discover_ground_truth():
        try: g=normalize_ground(read_table(p))
        except Exception: continue
        if g.station_name.str.startswith(("W","B"),na=False).any() and g["round"].isin(ROUNDS).any():
            compatible.append((p,g))
    if len(compatible)!=1:
        raise RuntimeError("Set GROUND_TRUTH_TABLE explicitly; discovery did not return exactly one compatible table.")
    ground_path,field=compatible[0]
else:
    ground_path=Path(GROUND_TRUTH_TABLE); field=normalize_ground(read_table(ground_path))
print("Ground truth:",ground_path)

rows=[]
for (station,rnd),g in field.groupby(["station_name","round"],observed=True):
    g=g.sort_values("wavelength_nm")
    for band,wl in SPECTRAL_WAVELENGTHS_NM.items():
        rows.append({"station_name":station,"round":int(rnd),"band_name":band,"wavelength_nm":wl,
                     "field_reflectance":np.interp(wl,g.wavelength_nm,g.field_reflectance)})
field_bands=pd.DataFrame(rows)
fw=field_bands.pivot_table(index=["station_name","round"],columns="band_name",
                           values="field_reflectance",aggfunc="mean").reset_index()
fw["field_ndvi740"]=(fw[NIR_BAND_NAME]-fw[RED_BAND_NAME])/(fw[NIR_BAND_NAME]+fw[RED_BAND_NAME])
field_ndvi=fw[["station_name","round","field_ndvi740"]]

## 11. Metrics

In [ ]:
spectra_comparison=spectra.merge(field_bands,on=["station_name","round","band_name","wavelength_nm"],how="inner")
ndvi_comparison=ndvi.merge(field_ndvi,on=["station_name","round"],how="inner")

def metrics(obs,pred):
    o,p=np.asarray(obs,float),np.asarray(pred,float)
    v=np.isfinite(o)&np.isfinite(p); o,p=o[v],p[v]
    if len(o)<2: return {"n":len(o),"r2":np.nan,"rmse":np.nan,"mae":np.nan,"bias":np.nan,"nrmse_range":np.nan}
    e=p-o; rmse=np.sqrt(np.mean(e*e)); rng=np.ptp(o)
    return {"n":len(o),"r2":r2_score(o,p),"rmse":rmse,"mae":np.mean(abs(e)),
            "bias":np.mean(e),"nrmse_range":rmse/rng if rng else np.nan}

def grouped(data,obs,pred,groups):
    out=[]
    for key,g in data.groupby(groups,observed=True):
        key=key if isinstance(key,tuple) else (key,)
        r=dict(zip(groups,key)); r.update(metrics(g[obs],g[pred])); out.append(r)
    return pd.DataFrame(out)

spectral_metrics=grouped(spectra_comparison,"field_reflectance","imagery_mean",
    ["round","site","dls_mode","scope_stage","route","absolute_option","band_name","wavelength_nm"])
spectral_overall=grouped(spectra_comparison,"field_reflectance","imagery_mean",
    ["round","site","dls_mode","scope_stage","route","absolute_option"])
ndvi_metrics=grouped(ndvi_comparison,"field_ndvi740","imagery_ndvi740_mean",
    ["round","site","dls_mode","scope_stage","route","absolute_option"])
display(spectral_overall.sort_values(["rmse","mae"]).head(30))
display(ndvi_metrics.sort_values(["rmse","mae"]).head(30))

## 13. Ground-truth comparison plots

The plots retain all stations and use larger, outlined markers for the
configured stations that should receive the most interpretation weight.

In [ ]:
best_spectral_cases = (
    spectral_overall.sort_values(["rmse", "mae"])
    .head(12)[
        ["round", "site", "dls_mode", "scope_stage", "route", "absolute_option"]
    ]
)

for case in best_spectral_cases.itertuples(index=False):
    group = spectra_comparison.loc[
        spectra_comparison["round"].eq(case.round)
        & spectra_comparison["site"].eq(case.site)
        & spectra_comparison["dls_mode"].eq(case.dls_mode)
        & spectra_comparison["scope_stage"].eq(case.scope_stage)
        & spectra_comparison["route"].eq(case.route)
        & spectra_comparison["absolute_option"].eq(case.absolute_option)
    ]
    if group.empty:
        continue

    fig, ax = plt.subplots(figsize=(7, 7))
    normal = group.loc[~group["highlight"]]
    focus = group.loc[group["highlight"]]

    ax.scatter(
        normal["field_reflectance"],
        normal["imagery_mean"],
        s=18, alpha=0.35, color="tab:blue", label="All other stations",
    )
    ax.scatter(
        focus["field_reflectance"],
        focus["imagery_mean"],
        s=48, alpha=0.9, facecolors="none",
        edgecolors="tab:red", linewidths=1.2,
        label="Highlighted stations",
    )

    values = np.r_[
        group["field_reflectance"].to_numpy(),
        group["imagery_mean"].to_numpy(),
    ]
    lo, hi = np.nanpercentile(values, [1, 99])
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Field reflectance")
    ax.set_ylabel("Imagery reflectance")
    ax.set_title(
        f"R{case.round} {case.site} {case.dls_mode}\n"
        f"{case.scope_stage} | {case.route} | {case.absolute_option}"
    )
    ax.legend()
    fig.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
for option, group in ndvi_comparison.groupby("absolute_option", observed=True):
    ax.scatter(
        group["field_ndvi740"],
        group["imagery_ndvi740_mean"],
        s=22, alpha=0.45, label=option,
    )
values = np.r_[
    ndvi_comparison["field_ndvi740"].to_numpy(),
    ndvi_comparison["imagery_ndvi740_mean"].to_numpy(),
]
lo, hi = np.nanpercentile(values, [1, 99])
ax.plot([lo, hi], [lo, hi], "k--", lw=1)
ax.set_xlabel("Field NDVI740")
ax.set_ylabel("Imagery NDVI740")
ax.set_title("NDVI740 comparison across absolute-calibration options")
ax.legend()
fig.tight_layout()
plt.show()

## 12. Save outputs

In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    chain_comparison.to_csv(OUTPUT_DIR/"chain_coefficient_comparison.csv",index=False)
    closure_summary.to_csv(OUTPUT_DIR/"chain_closure_uncertainty.csv",index=False)
    empirical_closure.to_csv(OUTPUT_DIR/"empirical_overlap_closure.csv",index=False)
    closure_draws.to_parquet(OUTPUT_DIR/"chain_closure_draws.parquet",index=False)
    station_pixels_raw.to_parquet(OUTPUT_DIR/"station_pixels_raw.parquet",index=False)
    spectra.to_parquet(OUTPUT_DIR/"station_spectra_mean_std.parquet",index=False)
    ndvi.to_parquet(OUTPUT_DIR/"station_ndvi740_mean_std.parquet",index=False)
    spectra_comparison.to_parquet(OUTPUT_DIR/"imagery_vs_field_spectra.parquet",index=False)
    ndvi_comparison.to_parquet(OUTPUT_DIR/"imagery_vs_field_ndvi740.parquet",index=False)
    spectral_metrics.to_csv(OUTPUT_DIR/"spectral_metrics_by_band.csv",index=False)
    spectral_overall.to_csv(OUTPUT_DIR/"spectral_metrics_overall.csv",index=False)
    ndvi_metrics.to_csv(OUTPUT_DIR/"ndvi740_metrics.csv",index=False)
print("Outputs:",OUTPUT_DIR)

## Review order

1. Inspect alignment shifts and overlap patches.
2. Reject suspicious spatial fits before using the coefficients.
3. Check closure gain against 1 and closure offset against 0, including intervals.
4. Compare the preferred and alternate Brunswick May 4 routes.
5. Compare no absolute calibration, legacy absolute calibration, and recomputed panel calibration.
6. Use RMSE, bias, and NDVI740 behavior in addition to R².
7. Keep all stations in tables; emphasize the configured stations because the others may have been trampled.